# Painel de Análise de Vendas

Atividade completa — etapas 1 a 20.

In [ ]:
# 1) Bibliotecas
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
os.makedirs("graficos", exist_ok=True)

In [ ]:
# 2) Carregar CSV e converter data
df = pd.read_csv("vendas.csv")
df["data"] = pd.to_datetime(df["data"], errors="coerce")
df.head()

In [ ]:
# 3) Criar faturamento
df["faturamento"] = df["quantidade"] * df["preco_unitario"]
df.head()

In [ ]:
# 4) Integridade
df.info()
display(df.isnull().sum())
display(df.describe().T)

In [ ]:
# 5) Preencher ausentes pela média
df["quantidade"] = df["quantidade"].fillna(df["quantidade"].mean())
df["preco_unitario"] = df["preco_unitario"].fillna(df["preco_unitario"].mean())
df["faturamento"] = df["quantidade"] * df["preco_unitario"]
display(df.isnull().sum())

In [ ]:
# 6) Faturamento por loja
faturamento_loja = df.groupby("loja")["faturamento"].sum().sort_values(ascending=False)
display(faturamento_loja.to_frame("faturamento_total"))

In [ ]:
# 7) Ticket médio por loja
ticket_medio = df.groupby("loja").agg(faturamento_total=("faturamento","sum"),
                                      numero_vendas=("faturamento","count"))
ticket_medio["ticket_medio"] = ticket_medio["faturamento_total"] / ticket_medio["numero_vendas"]
display(ticket_medio.sort_values("ticket_medio", ascending=False))

In [ ]:
# 8) Produtos mais vendidos
produtos_mais_vendidos = df.groupby("produto")["quantidade"].sum().sort_values(ascending=False)
display(produtos_mais_vendidos.to_frame("quantidade_total"))

In [ ]:
# 9) Maior e menor mês
faturamento_mensal = df.groupby(df["data"].dt.to_period("M"))["faturamento"].sum().sort_index()
print("Maior:", faturamento_mensal.idxmax().strftime("%m/%Y"), f"— R$ {faturamento_mensal.max():,.2f}")
print("Menor:", faturamento_mensal.idxmin().strftime("%m/%Y"), f"— R$ {faturamento_mensal.min():,.2f}")

In [ ]:
# 10) Coluna mês
df["mes"] = df["data"].dt.month
df["mes_ano"] = df["data"].dt.to_period("M").astype(str)
display(df[["data","mes","mes_ano"]].head())

In [ ]:
# Preparação para as análises de região/categoria
mapa_regiao = {"Loja Centro":"Sudeste","Loja Norte":"Norte","Loja Sul":"Sul","Loja Oeste":"Centro-Oeste"}
mapa_categoria = {"Notebook":"Informática","Smartphone":"Eletrônicos","Tablet":"Eletrônicos","Monitor":"Informática",
                  "Teclado":"Periféricos","Mouse":"Periféricos","Headset":"Periféricos","Impressora":"Informática"}
df["regiao"] = df["loja"].map(mapa_regiao)
df["categoria"] = df["produto"].map(mapa_categoria)

# 17) Tema global
sns.set_style("whitegrid")
sns.set_palette("deep")

In [ ]:
# 11) Barras — faturamento por loja
dados = faturamento_loja.sort_values(ascending=False)
cores = sns.color_palette("deep", n_colors=len(dados))
plt.figure(figsize=(10,6))
plt.bar(dados.index, dados.values, color=cores)
plt.title("Faturamento Total por Loja")
plt.xlabel("Loja"); plt.ylabel("Faturamento (R$)")
plt.xticks(rotation=20)
for i, v in enumerate(dados.values):
    plt.text(i, v, f"R$ {v:,.0f}", ha="center", va="bottom")
plt.tight_layout()
plt.savefig("graficos/01_faturamento_por_loja.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 12) Linhas — evolução mensal
serie = faturamento_mensal
x = serie.index.astype(str)
plt.figure(figsize=(11,6))
plt.plot(x, serie.values, marker="o", linewidth=2)
imax, imin = int(np.argmax(serie.values)), int(np.argmin(serie.values))
plt.scatter([x[imax]],[serie.iloc[imax]],s=90)
plt.scatter([x[imin]],[serie.iloc[imin]],s=90)
plt.annotate(f"Maior: R$ {serie.iloc[imax]:,.0f}", (x[imax],serie.iloc[imax]), xytext=(0,18), textcoords="offset points", ha="center")
plt.annotate(f"Menor: R$ {serie.iloc[imin]:,.0f}", (x[imin],serie.iloc[imin]), xytext=(0,-28), textcoords="offset points", ha="center")
plt.title("Evolução Mensal do Faturamento")
plt.xlabel("Mês/Ano"); plt.ylabel("Faturamento (R$)")
plt.xticks(rotation=30); plt.tight_layout()
plt.savefig("graficos/02_evolucao_mensal.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 13) Pizza — faturamento por região
faturamento_regiao = df.groupby("regiao")["faturamento"].sum().sort_values(ascending=False)
plt.figure(figsize=(8,8))
plt.pie(faturamento_regiao.values, labels=faturamento_regiao.index, autopct="%1.1f%%", startangle=90)
plt.title("Distribuição do Faturamento por Região")
plt.tight_layout()
plt.savefig("graficos/03_faturamento_por_regiao.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 14) Dispersão — quantidade x faturamento
dados_produto = df.groupby("produto").agg(quantidade=("quantidade","sum"),
                                           faturamento=("faturamento","sum")).reset_index()
plt.figure(figsize=(10,6))
sns.scatterplot(data=dados_produto, x="quantidade", y="faturamento", hue="produto", s=120)
plt.title("Relação entre Quantidade Vendida e Faturamento por Produto")
plt.xlabel("Quantidade Vendida"); plt.ylabel("Faturamento (R$)")
plt.legend(title="Produto", bbox_to_anchor=(1.02,1), loc="upper left")
plt.tight_layout()
plt.savefig("graficos/04_quantidade_vs_faturamento.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 15) Boxplot — preços por categoria
plt.figure(figsize=(10,6))
sns.boxplot(data=df, x="categoria", y="preco_unitario")
plt.title("Distribuição dos Preços por Categoria de Produto")
plt.xlabel("Categoria"); plt.ylabel("Preço Unitário (R$)")
plt.xticks(rotation=15); plt.tight_layout()
plt.savefig("graficos/05_boxplot_precos_categoria.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 16) Heatmap — correlação
correlacao = df[["quantidade","preco_unitario","faturamento"]].corr()
plt.figure(figsize=(8,6))
sns.heatmap(correlacao, annot=True, fmt=".2f", cmap="Blues", linewidths=.5)
plt.title("Matriz de Correlação das Variáveis Numéricas")
plt.tight_layout()
plt.savefig("graficos/06_heatmap_correlacao.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 18) Revisão dos gráficos
# Cada gráfico possui título descritivo, rótulos dos eixos,
# legenda quando necessária e layout ajustado.

In [ ]:
# 19) Layout lado a lado
fig, axes = plt.subplots(1,2,figsize=(16,6))
faturamento_loja.plot(kind="bar", ax=axes[0], title="Faturamento por Loja")
axes[0].set_xlabel("Loja"); axes[0].set_ylabel("Faturamento (R$)")
faturamento_regiao.plot(kind="bar", ax=axes[1], title="Faturamento por Região")
axes[1].set_xlabel("Região"); axes[1].set_ylabel("Faturamento (R$)")
plt.suptitle("Comparação de Faturamento por Loja e por Região", fontsize=16)
plt.tight_layout()
plt.savefig("graficos/07_comparativo_loja_regiao.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 20) Exportação e relatório final
resumo = pd.DataFrame({
    "indicador": ["Loja com maior faturamento","Loja com menor faturamento",
                  "Produto mais vendido em quantidade","Mês de maior faturamento",
                  "Mês de menor faturamento","Região com maior faturamento"],
    "resultado": [faturamento_loja.idxmax(), faturamento_loja.idxmin(),
                  produtos_mais_vendidos.idxmax(), faturamento_mensal.idxmax().strftime("%m/%Y"),
                  faturamento_mensal.idxmin().strftime("%m/%Y"), faturamento_regiao.idxmax()],
    "valor": [faturamento_loja.max(), faturamento_loja.min(), produtos_mais_vendidos.max(),
              faturamento_mensal.max(), faturamento_mensal.min(), faturamento_regiao.max()]
})
resumo.to_csv("resumo_analitico.csv", index=False)
display(resumo)

print("Comentários analíticos:")
print(f"- Maior faturamento por loja: {faturamento_loja.idxmax()}." )
print(f"- Produto mais vendido em quantidade: {produtos_mais_vendidos.idxmax()}." )
print(f"- Maior faturamento mensal: {faturamento_mensal.idxmax().strftime('%m/%Y')}." )
print(f"- Menor faturamento mensal: {faturamento_mensal.idxmin().strftime('%m/%Y')}." )
print(f"- Região com maior faturamento: {faturamento_regiao.idxmax()}." )